# LangGraphユーザーのためのGraflow入門 〜比較で学ぶハンズオンガイド〜

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GraflowAI/graflow/blob/main/docs/tutorials/langgraph_comparison_tutorial.ipynb)

このノートブックは、Zenn記事「[LangGraphユーザーのためのGraflow入門](https://zenn.dev/myui/articles/)」で紹介されているコード例を、Google Colabで実際に動かしながら学べるチュートリアルです。

## このチュートリアルで学べること

| # | トピック | 内容 |
|---|---|---|
| 1 | 最初のグラフ | `@task` + `>>` 演算子で最もシンプルなワークフロー |
| 2 | 並列実行 | `\|` 演算子でDiamondパターン（Fan-out → Fan-in） |
| 3 | データ共有 | Channelによるタスク間データ共有 + 自動キーワード引数解決 |
| 4 | 条件分岐・ループ | `next_task()` / `next_iteration()` / `max_cycles` / `RetryPolicy` |
| 5 | 並列エラーポリシー | Best-effort / At-least-N / Critical ポリシー |
| 6 | 総合ハンズオン | データ分析パイプライン（すべての要素を統合） |
| 7-11 | LLM統合 | `inject_llm_client` / `inject_llm_agent` によるLLM連携 |

> **Note**: 分散実行（Redis）、トレーシング（Langfuse）は外部サービスが必要なため、本ノートブックではコード紹介のみとしています。LLM統合（`inject_llm_client`）は `OPENAI_API_KEY` を設定すれば実行可能です。

## 0. 環境構築

In [ ]:
!pip install -q graflow

In [ ]:
# バージョン確認
import graflow

print(f"Graflow version: {graflow.__version__}")

---

## 1. 最初のグラフを作る — Stateの事前定義は不要

LangGraphでは `TypedDict` でStateを定義 → `add_node` / `add_edge` でグラフ組み立て → `compile()` → `invoke()` と4ステップ必要です。

Graflowでは:
1. `@task` で関数をタスク化
2. `>>` 演算子で依存関係を定義
3. `ctx.execute()` で実行

**Stateの事前定義（TypedDict）、`add_node`/`add_edge`の個別呼び出し、`compile()` ステップが不要**です。

In [ ]:
from graflow.core.context import TaskExecutionContext
from graflow.core.decorators import task
from graflow.core.workflow import workflow

with workflow("my_pipeline") as ctx:

    @task(inject_context=True)
    def task_a(context: TaskExecutionContext):
        channel = context.get_channel()
        text = channel.get("text", "")
        channel.set("text", text + "a")

    @task(inject_context=True)
    def task_b(context: TaskExecutionContext):
        channel = context.get_channel()
        text = channel.get("text", "")
        channel.set("text", text + "b")

    # >> 演算子で順序を定義
    task_a >> task_b

    # 実行（ret_context=True でチャンネルにアクセス可能）
    _, exec_ctx = ctx.execute("task_a", ret_context=True)
    print(exec_ctx.get_channel().get("text"))  # 'ab'

### Low-Level API: `add_node` / `add_edge` スタイル

LangGraphのような Low-Level API もサポートしています。動的なグラフ構築や細かい制御が必要な場面で使えます。

In [ ]:
from graflow.core.context import ExecutionContext
from graflow.core.engine import WorkflowEngine
from graflow.core.graph import TaskGraph
from graflow.core.task import TaskWrapper

# LangGraph風にグラフを構築
graph = TaskGraph()

extract = TaskWrapper("extract", func=lambda: print("  extract"), register_to_context=False)
transform = TaskWrapper("transform", func=lambda: print("  transform"), register_to_context=False)
load = TaskWrapper("load", func=lambda: print("  load"), register_to_context=False)

graph.add_node(extract, "extract")
graph.add_node(transform, "transform")
graph.add_node(load, "load")

graph.add_edge("extract", "transform")
graph.add_edge("transform", "load")

# 実行
context = ExecutionContext.create(graph, "extract", max_steps=10)
engine = WorkflowEngine()
engine.execute(context)

---

## 2. 並列実行 — 1行で構造が読める

LangGraphでは分岐元から複数ノードにエッジを張り、合流先への依存を設定します。ノード数が増えるほど `add_edge` の行数も増えます。

Graflowでは `>>` が直列、`|` が並列。**Diamond（Fan-out → Fan-in）パターンが1行**で書けます。

```
fetch >> (transform_a | transform_b) >> store
```

In [ ]:
from graflow.core.decorators import task
from graflow.core.workflow import workflow

with workflow("diamond") as ctx:

    @task
    def fetch():
        print("  データ取得")

    @task
    def transform_a():
        print("  変換A")

    @task
    def transform_b():
        print("  変換B")

    @task
    def store():
        print("  保存")

    # Diamond パターンが1行で表現できる
    fetch >> (transform_a | transform_b) >> store

    ctx.execute("fetch")

### 関数スタイルによる動的タスクリスト構築

`chain()` / `parallel()` 関数を使って、動的にタスクリストを構築することもできます。

In [ ]:
from graflow.core.decorators import task
from graflow.core.task import parallel
from graflow.core.workflow import workflow

with workflow("dynamic_parallel") as ctx:

    @task
    def start():
        print("  開始")

    # 動的にタスクを生成
    worker_tasks = []
    for i in range(4):

        @task(id=f"worker_{i}")
        def worker(task_id=i):
            print(f"  Worker {task_id} 実行中")

        worker_tasks.append(worker)

    @task
    def aggregate():
        print("  結果を集約")

    # chain + parallel で動的に構築
    start >> parallel(*worker_tasks) >> aggregate

    ctx.execute("start")

---

## 3. タスク間のデータ共有 — Reducerは不要

LangGraphでは `State`（TypedDict）を共有し、複数ノードからの更新をどうマージするかをReducerで制御する必要があります。

Graflowでは **Channel**（Key-Valueストア）でタスク間のデータを明示的に読み書きします。「何をいつ更新するか」はタスクの中で制御するため、暗黙的なマージルールであるReducerという概念自体が不要です。

デフォルトの **MemoryChannel** はインメモリ実装でスレッドセーフではありませんが、**「書き込みは自タスク（ノードID）のキーに対して行い、読み込みは自由」** というルールを守れば、並列実行でも競合は発生しません。これがChannelのベストプラクティスです。

**RedisChannel** バックエンドを利用した場合は、Redisの非同期I/O（ノンブロッキングI/O＋イベントループ）によるシングルスレッド動作により、複数エージェント（ワーカー）からの読み書きはRedis側でリクエストが直列化されます。アプリケーション側でロックや排他制御といった**同期制御は不要**です。

### Channel の基本: `set` / `get`

In [ ]:
from graflow.core.context import TaskExecutionContext
from graflow.core.decorators import task
from graflow.core.workflow import workflow

with workflow("channel_demo") as ctx:

    @task(inject_context=True)
    def producer(context: TaskExecutionContext):
        channel = context.get_channel()
        channel.set("config", {"batch_size": 100})
        channel.set("counter", 1)
        print(f"  Producer: config={channel.get('config')}, counter={channel.get('counter')}")

    @task(inject_context=True)
    def consumer(context: TaskExecutionContext):
        channel = context.get_channel()
        config = channel.get("config")  # {"batch_size": 100}
        counter = channel.get("counter")  # 1
        channel.set("counter", counter + 1)  # 明示的に更新
        print(f"  Consumer: config={config}, counter={channel.get('counter')}")

    producer >> consumer
    ctx.execute("producer")

### 自動キーワード引数解決

`inject_context` すら不要！タスクの引数名がチャンネルのキーと自動的にマッチングされます。

In [ ]:
from graflow.core.context import TaskExecutionContext
from graflow.core.decorators import task
from graflow.core.workflow import workflow

with workflow("auto_resolve") as ctx:

    @task(inject_context=True)
    def setup(context: TaskExecutionContext):
        channel = context.get_channel()
        channel.set("user_name", "Alice")
        channel.set("user_city", "Tokyo")

    @task
    def greet(user_name: str, user_city: str = "Unknown"):
        # inject_context 不要！引数名がチャンネルのキーと自動一致
        print(f"  Hello, {user_name} from {user_city}!")

    setup >> greet
    ctx.execute("setup")

### TypedChannel — 型安全なデータ共有

`TypedDict` でスキーマを定義し、`get_typed_channel()` で型付きチャンネルを取得すると、IDEの**コード補完やタイプチェック**が有効になります。キー名のタイプミスや型の不整合をコーディング時に検出でき、大規模なワークフローでも安全にデータを扱えます。

In [ ]:
from typing import TypedDict

from graflow.core.context import TaskExecutionContext
from graflow.core.decorators import task
from graflow.core.workflow import workflow


class UserProfile(TypedDict):
    user_id: str
    name: str
    email: str


with workflow("typed_channel_demo") as ctx:

    @task(inject_context=True)
    def collect_user(context: TaskExecutionContext):
        profile_ch = context.get_typed_channel(UserProfile)
        profile_ch.set(
            "current_user",
            {
                "user_id": "u_123",
                "name": "Alice",
                "email": "alice@example.com",
            },
        )
        print("  ユーザー情報を保存しました")

    @task(inject_context=True)
    def display_user(context: TaskExecutionContext):
        profile_ch = context.get_typed_channel(UserProfile)
        user = profile_ch.get("current_user")
        print(f"  ユーザー: {user['name']} ({user['email']})")

    collect_user >> display_user
    ctx.execute("collect_user")

---

## 4. 条件分岐とループ — 事前定義不要の動的制御

ここがLangGraphとGraflowで**最も設計思想が異なるポイント**です。

- **LangGraph**: `add_conditional_edges` ですべての分岐パスを**コンパイル時に固定**
- **Graflow**: `next_task()` / `next_iteration()` で**実行時に動的に制御**

### `next_iteration()` — リトライ（自分自身を再実行）

In [ ]:
import random

from graflow.core.context import TaskExecutionContext
from graflow.core.decorators import task
from graflow.core.workflow import workflow

with workflow("retry_demo") as ctx:

    @task(inject_context=True)
    def process_data(context: TaskExecutionContext):
        channel = context.get_channel()
        attempt = channel.get("attempt", 0) + 1
        channel.set("attempt", attempt)

        # ランダムなスコアをシミュレーション（試行回数が増えると成功しやすくなる）
        score = random.random() * (0.5 + attempt * 0.15)
        score = min(score, 1.0)
        print(f"  試行 {attempt}: score = {score:.2f}")

        if score < 0.8 and attempt < 5:
            # リトライ: 自分自身を再実行
            print("  → スコアが低いためリトライ")
            context.next_iteration()
        else:
            channel.set("final_score", score)
            print(f"  → 成功! 最終スコア: {score:.2f}")

    @task
    def finalize(final_score: float):
        print(f"  完了: 最終スコア = {final_score:.2f}")

    process_data >> finalize

    random.seed(42)
    ctx.execute("process_data")

### `next_task()` — 動的分岐（新しいタスクを実行時に生成）

LangGraphの `Send` は「事前定義済みノードへの動的エッジ」ですが、Graflowの `next_task()` は**新しいタスクの生成＋グラフへの追加**を実行時に行います。

In [ ]:
from graflow.core.context import TaskExecutionContext
from graflow.core.decorators import task
from graflow.core.task import TaskWrapper
from graflow.core.workflow import workflow

with workflow("dynamic_fanout") as ctx:

    @task(inject_context=True)
    def discover_files(context: TaskExecutionContext):
        """実行時にファイル数が判明し、それに応じてタスクを動的生成"""
        files = ["report.csv", "users.json", "logs.txt"]
        print(f"  {len(files)} 個のファイルを発見")

        # ファイル数に応じてタスクを動的生成（Fan-out）
        for file in files:
            context.next_task(
                TaskWrapper(
                    f"process_{file}",
                    lambda f=file: print(f"  処理中: {f}"),
                )
            )

    discover_files  # 起点タスクのみ定義。後続は実行時に動的生成

    ctx.execute("discover_files")

### 動的制御メソッド一覧

| メソッド | 動作 | 用途 |
|---|---|---|
| `next_task(task)` | 新しいタスクを追加 | 動的分岐、Fan-out |
| `next_task(task, goto=True)` | 既存タスクにジャンプ（後続スキップ） | 早期脱出、エラーハンドリング |
| `next_iteration()` | 自分自身を再実行 | リトライ、収束ループ |
| `terminate_workflow()` | 正常終了 | 早期完了 |
| `cancel_workflow(reason)` | 異常終了 | エラー時のキャンセル |

### `max_cycles` — イテレーション回数の制御

`next_iteration()` による手動ループに加え、`@task(max_cycles=N)` でイテレーション上限を宣言的に設定できます。`ctx.can_iterate()` で残りサイクルがあるかを確認し、`ctx.cycle_count` で現在のサイクル番号（1始まり）を取得できます。

LangGraphでループを実現するには `add_conditional_edges` でグラフにサイクルを事前定義する必要がありますが、Graflowでは **タスク単体で完結** します。

In [ ]:
from graflow.core.context import ExecutionContext, TaskExecutionContext
from graflow.core.decorators import task
from graflow.core.engine import WorkflowEngine
from graflow.core.graph import TaskGraph

graph = TaskGraph()

@task(inject_context=True, max_cycles=20)
def optimizer(ctx: TaskExecutionContext, data=None):
    loss = (data or {}).get("loss", 1.0) * 0.5
    print(f"  cycle {ctx.cycle_count}: loss={loss:.4f}")
    if loss < 0.05:
        print(f"  Converged at cycle {ctx.cycle_count}")
        return
    if ctx.can_iterate():
        ctx.next_iteration({"loss": loss})

graph.add_node(optimizer, "optimizer")
WorkflowEngine().execute(
    ExecutionContext.create(graph, start_node="optimizer")
)

### `RetryPolicy` — 失敗時の自動リトライ

LangGraphでも `RetryPolicy` によるノード単位のリトライが可能ですが、Graflowでは `@task` デコレータで直接指定でき、`jitter`（ランダム化）や `max_interval`（上限キャップ）といった追加オプションも利用できます。`default_max_retries` でワークフロー全体のデフォルトも設定可能です。

In [ ]:
from graflow.core.context import ExecutionContext
from graflow.core.decorators import task
from graflow.core.engine import WorkflowEngine
from graflow.core.graph import TaskGraph
from graflow.core.retry import RetryPolicy
from graflow.exceptions import GraflowRuntimeError

graph = TaskGraph()
attempts = 0

@task(
    retry_policy=RetryPolicy(
        max_retries=3,
        initial_interval=0.1,  # short for demo
        backoff_factor=2.0,    # 0.1s → 0.2s → 0.4s
    ),
)
def flaky_service():
    global attempts
    attempts += 1
    print(f"  attempt {attempts}")
    if attempts < 3:
        raise ConnectionError(f"timeout (attempt {attempts})")
    return "success"

graph.add_node(flaky_service, "flaky_service")
context = ExecutionContext.create(graph, start_node="flaky_service")
WorkflowEngine().execute(context)
print(f"  Recovered after {attempts} attempts, result: {context.get_result('flaky_service')}")

---

## 5. 並列グループのエラーポリシー

LangGraphには並列実行時のエラーハンドリングポリシーがありません。Graflowでは**並列グループ単位で柔軟にエラー制御**が可能です。

| ポリシー | 動作 | 適用例 |
|---|---|---|
| **Strict**（デフォルト） | 全タスク成功必須 | 金融取引、データ検証 |
| **Best-effort** | 部分成功で続行 | 通知送信、オプション処理 |
| **At-least-N** | 最小成功数を指定 | マルチリージョンデプロイ |
| **Critical** | 重要タスクのみ判定 | 必須+オプションの混在 |

In [ ]:
from graflow.coordination.executor import CoordinationBackend
from graflow.core.decorators import task
from graflow.core.handlers.group_policy import (
    BestEffortGroupPolicy,
)
from graflow.core.workflow import workflow

with workflow("error_policy_demo") as ctx:

    @task
    def send_email():
        print("  Email 送信成功")

    @task
    def send_sms():
        raise RuntimeError("SMS送信失敗!")  # わざと失敗させる

    @task
    def send_slack():
        print("  Slack 送信成功")

    @task
    def done():
        print("  通知処理完了（一部失敗しても続行）")

    # BestEffortPolicy: 一部失敗しても続行
    (send_email | send_sms | send_slack).with_execution(
        backend=CoordinationBackend.THREADING,
        policy=BestEffortGroupPolicy(),
    ) >> done

    ctx.execute("send_email")

---

## 6. 総合ハンズオン 〜データ分析パイプライン〜

これまでの要素を組み合わせた実践的なパイプラインです。

- `@task` デコレータと `>>` / `|` 演算子
- Channel によるタスク間データ共有
- 自動キーワード引数解決（`generate_report` の引数がチャンネルから自動取得）
- Diamond パターン（Fan-out → Fan-in）

In [ ]:
from graflow.core.context import TaskExecutionContext
from graflow.core.decorators import task
from graflow.core.workflow import workflow

with workflow("data_analysis") as ctx:

    @task(inject_context=True)
    def fetch_data(context: TaskExecutionContext):
        """データを取得してチャンネルに格納"""
        data = {
            "sales": [100, 200, 150, 300, 250],
            "costs": [50, 80, 60, 120, 100],
        }
        channel = context.get_channel()
        channel.set("raw_data", data)
        print(f"📥 データ取得: {len(data['sales'])}件")

    @task(inject_context=True)
    def analyze_sales(context: TaskExecutionContext):
        """売上分析（並列タスク1）"""
        channel = context.get_channel()
        sales = channel.get("raw_data")["sales"]
        total = sum(sales)
        channel.set("sales_total", total)
        print(f"📊 売上分析: 合計={total}, 平均={total / len(sales)}")

    @task(inject_context=True)
    def analyze_costs(context: TaskExecutionContext):
        """コスト分析（並列タスク2）"""
        channel = context.get_channel()
        costs = channel.get("raw_data")["costs"]
        total = sum(costs)
        channel.set("cost_total", total)
        print(f"💰 コスト分析: 合計={total}, 平均={total / len(costs)}")

    @task
    def generate_report(sales_total: int, cost_total: int):
        """分析結果を統合してレポート生成（自動キーワード引数解決）"""
        profit = sales_total - cost_total
        margin = (profit / sales_total) * 100
        print("\n📝 === 分析レポート ===")
        print(f"   売上合計: {sales_total}")
        print(f"   コスト合計: {cost_total}")
        print(f"   利益: {profit} （利益率: {margin:.1f}%）")

    # ワークフロー定義（1行で構造が読める）
    fetch_data >> (analyze_sales | analyze_costs) >> generate_report

    ctx.execute("fetch_data")

---

## 7. Checkpoint/Resume（コード紹介）

> **Note**: チェックポイントはファイルシステムに保存するため、Colabでも動作しますが、Resume のシナリオは実運用で効果を発揮します。

LangGraphのチェックポイントは各ステップで**自動保存**されますが、State全体のシリアライズは重い処理です。Graflowでは**ユーザーが重要なポイントだけを選んで明示的に保存**できます。

```python
from graflow.core.checkpoint import CheckpointManager

@task(inject_context=True)
def train_model(context):
    for epoch in range(100):
        loss = train_one_epoch(epoch)

        if epoch % 10 == 0:
            # 重要なポイントで明示的に保存
            context.checkpoint(
                path="/tmp/ml_checkpoint",
                metadata={"epoch": epoch, "loss": loss}
            )

# 障害発生後: 最後のチェックポイントから再開
CheckpointManager.resume_from_checkpoint("/tmp/ml_checkpoint")
```

---

## 8. Human-in-the-Loop（コード紹介）

> **Note**: HITL は実運用でSlack Webhook等と連携して使うことを想定しています。

LangGraphの `interrupt()` は例外を送出してグラフを中断する仕組みです。Graflowの `request_feedback()` は**通常の関数呼び出しとして応答を待機**し、タイムアウト時には自動でチェックポイントを保存してプロセスを解放します。

```python
@task(inject_context=True)
def request_deployment_approval(context):
    response = context.request_feedback(
        feedback_type="approval",
        prompt="本番環境へのデプロイを承認しますか？",
        timeout=300,  # 5分待機
        notification_config={
            "type": "webhook",
            "url": "https://hooks.slack.com/services/XXX",
            "message": "デプロイ承認が必要です"
        }
    )

    if response.approved:
        print("✅ 承認されました")
    else:
        context.cancel_workflow("承認が拒否されました")
```

**動作フロー:**
1. `request_feedback()` を呼び出し → 人間に通知
2. タイムアウト内に承認 → **そのまま続行**（Blockingモード）
3. タイムアウト経過 → **自動でチェックポイント保存、プロセスを解放**
4. 後日API経由で承認 → **チェックポイントから再開**

---

## 9. 分散実行（コード紹介）

> **Note**: 分散実行にはRedisが必要です。ローカルやKubernetes環境で利用してください。

LangGraphはシングルプロセス実行のみですが、Graflowは**Redisベースの分散ワーカー**をOSSとして標準装備しています。

```bash
# Step 1: Redis起動
docker run -p 6379:6379 redis:7.2

# Step 2: ワーカーを複数起動
python -m graflow.worker.main --worker-id worker-1 --redis-host localhost
python -m graflow.worker.main --worker-id worker-2 --redis-host localhost
```

```python
from graflow.coordination.executor import CoordinationBackend

# この1行だけでローカル → 分散に切り替え
parallel = (task_a | task_b | task_c).with_execution(
    backend=CoordinationBackend.REDIS,
    backend_config={"redis_client": redis_client}
)
```

---

## 10. タスクハンドラー（コード紹介）

> **Note**: Dockerハンドラーの実行にはDockerデーモンが必要です。

LangGraphのノードはプロセス内実行のみですが、Graflowではタスクごとに**実行戦略（ハンドラー）を切り替え**られます。

```python
# デフォルト: プロセス内実行
@task
def simple_task():
    return "result"

# Docker コンテナ内で実行（GPU、依存関係の隔離）
@task(handler="docker", handler_kwargs={
    "image": "pytorch/pytorch:2.0-gpu",
    "gpu": True,
    "volumes": {"/data": "/workspace/data"},
})
def train_on_gpu():
    return train_model()
```

| ハンドラー | 説明 | 用途 |
|---|---|---|
| `direct` | プロセス内実行（デフォルト） | 一般的なタスク |
| `docker` | コンテナ実行 | GPU処理、サンドボックス実行 |
| カスタム | 自由に実装可能 | Cloud Run、リモート実行など |

---

## 11. LLM統合 — `inject_llm_client` / `inject_llm_agent`

Graflowは特定のLLMフレームワークに依存せず、2つの統合方式を提供します。

| 方式 | デコレータ | 用途 |
|---|---|---|
| **LiteLLM統合** | `@task(inject_llm_client=True)` | OpenAI / Claude / Gemini 等を統一APIで呼び出し |
| **SuperAgent** | `@task(inject_llm_agent="name")` | Google ADK等のエージェントフレームワークを依存性注入 |

### 方式1: `inject_llm_client`（LiteLLM統合）

`@task(inject_llm_client=True)` を指定すると、タスクの第1引数に `LLMClient` が自動注入されます。
LLMClientは [LiteLLM](https://docs.litellm.ai/) のラッパーで、OpenAI / Claude / Gemini など100以上のLLMプロバイダーを統一APIで利用できます。

**ポイント:**
- `LLMClient` はワークフロー内で**シングルトン**（全タスクで同一インスタンスを共有）
- デフォルトモデルを設定しつつ、呼び出しごとに別モデルを指定可能
- `OPENAI_API_KEY` 等の環境変数を設定するだけで動作
- `inject_llm_client=True` 指定時に `LLMClient` が未登録の場合、デフォルトモデル（`gpt-5-mini` または `GRAFLOW_LLM_MODEL` 環境変数）で自動生成

In [ ]:
!pip install -q litellm

import os

# Google Colab の場合: 左メニューからシークレットに OPENAI_API_KEY を追加
# または直接設定:
# os.environ["OPENAI_API_KEY"] = "sk-..."

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("OPENAI_API_KEY loaded from Colab secrets")
except Exception:
    if "OPENAI_API_KEY" in os.environ:
        print("OPENAI_API_KEY found in environment")
    else:
        print("WARNING: OPENAI_API_KEY not set. Set it to run LLM examples.")

In [ ]:
from graflow.core.decorators import task
from graflow.core.workflow import workflow
from graflow.llm.client import LLMClient

with workflow("llm_client_demo") as ctx:
    # LLMClient をワークフローに登録（全タスクで共有）
    llm_client = LLMClient(model="gpt-5-mini", enable_tracing=False)
    ctx.register_llm_client(llm_client)

    @task(inject_llm_client=True)
    def summarize(llm_client: LLMClient) -> str:
        """LLMClient が第1引数に自動注入される"""
        return llm_client.completion_text(
            messages=[{"role": "user", "content": "Pythonを一言で説明してください"}],
        )

    @task(inject_llm_client=True)
    def translate(llm_client: LLMClient, summary: str) -> str:
        """Channel から summary を自動解決 + LLMClient 注入"""
        return llm_client.completion_text(
            messages=[{"role": "user", "content": f"Translate to English: {summary}"}],
        )

    # summarize の戻り値は自動的に Channel に格納され、
    # translate の引数 summary に自動解決される
    summarize >> translate

    result = ctx.execute("summarize")

print(f"翻訳結果: {result}")

#### LLMClient の自動生成

`register_llm_client()` を呼ばなくても、`inject_llm_client=True` のタスクが実行されると **デフォルトの LLMClient が自動生成**されます。
デフォルトモデルは `gpt-5-mini`（環境変数 `GRAFLOW_LLM_MODEL` で変更可能）です。

In [ ]:
from graflow.core.decorators import task
from graflow.core.workflow import workflow
from graflow.llm.client import LLMClient

with workflow("llm_auto_demo") as ctx:
    # register_llm_client() を呼ばない → デフォルトで gpt-5-mini が使われる

    @task(inject_llm_client=True)
    def ask(llm_client: LLMClient) -> str:
        print(f"Auto-created model: {llm_client.model}")
        return llm_client.completion_text(
            messages=[{"role": "user", "content": "What is 1+1? Answer in one word."}],
        )

    result = ctx.execute("ask")

print(f"回答: {result}")

#### 呼び出しごとのモデル切り替え

同一タスク内で、呼び出しごとに異なるモデルを使い分けることも可能です。

In [ ]:
from graflow.core.decorators import task
from graflow.core.workflow import workflow
from graflow.llm.client import LLMClient

with workflow("llm_model_switch") as ctx:
    llm_client = LLMClient(model="gpt-5-mini", enable_tracing=False)
    ctx.register_llm_client(llm_client)

    @task(inject_llm_client=True)
    def multi_model(llm_client: LLMClient) -> dict:
        # デフォルトモデル（gpt-5-mini）で高速に処理
        quick = llm_client.completion_text(
            messages=[{"role": "user", "content": "Say hello in Japanese"}],
        )
        # 別モデルを指定して呼び出し
        detailed = llm_client.completion_text(
            messages=[{"role": "user", "content": "Explain quantum computing in one sentence"}],
            model="gpt-4o-mini",  # この呼び出しだけ別モデル
        )
        return {"quick": quick, "detailed": detailed}

    result = ctx.execute("multi_model")

print(f"gpt-5-mini: {result['quick']}")
print(f"gpt-4o-mini: {result['detailed']}")

### 方式2: `inject_llm_agent`（SuperAgentの依存性注入）

> **Note**: `inject_llm_agent` は Google ADK 等のエージェントフレームワークとの連携機能です。ADK のインストールが必要なため、ここではコード紹介のみとします。

`@task(inject_llm_agent="agent_name")` を指定すると、事前登録したエージェントがタスクに注入されます。
エージェントは Graflow の `LLMAgent` 基底クラスを実装したラッパー（例: `AdkLLMAgent`）で、ReActループやツール呼び出しを含む複雑なLLMインタラクションをカプセル化します。

```python
from google.adk.agents import LlmAgent
from graflow.core.decorators import task
from graflow.core.workflow import workflow
from graflow.llm.agents.adk_agent import AdkLLMAgent

with workflow("agent_demo") as ctx:
    # ADK エージェントを作成してラップ
    adk_agent = LlmAgent(
        name="researcher",
        model="gemini-2.5-flash",
        instruction="You are a research assistant.",
        tools=[search_tool, calculator_tool],
    )
    ctx.register_llm_agent("researcher", AdkLLMAgent(adk_agent))

    @task(inject_llm_agent="researcher")
    def research(llm_agent, query: str) -> str:
        """登録名で指定したエージェントが自動注入される"""
        result = llm_agent.run(query)
        return result["output"]

    @task(inject_llm_client=True, inject_llm_agent="researcher")
    def analyze(llm_client, llm_agent, data: str) -> dict:
        """LLMClient と LLMAgent を同時に注入することも可能"""
        summary = llm_client.completion_text(
            messages=[{"role": "user", "content": f"Summarize: {data}"}],
        )
        deep_analysis = llm_agent.run(f"Analyze in detail: {data}")
        return {"summary": summary, "analysis": deep_analysis["output"]}
```

**ポイント:**
- `register_llm_agent()` にはファクトリ関数も渡せる（遅延初期化）
- 複数のエージェントを名前で区別して登録・注入可能
- `inject_llm_client` と `inject_llm_agent` は同一タスクで併用可能
- `inject_context=True` との3つ同時注入も対応

### LLM統合の比較

| 観点 | `inject_llm_client` | `inject_llm_agent` |
|---|---|---|
| **抽象度** | 低レベル（completion API） | 高レベル（エージェントループ） |
| **用途** | プロンプト → レスポンスの単純な呼び出し | ツール使用・マルチステップ推論 |
| **バックエンド** | LiteLLM（100+プロバイダー対応） | Google ADK / PydanticAI 等 |
| **インスタンス** | ワークフロー共有シングルトン | 名前付きで複数登録可能 |
| **自動生成** | 未登録時にデフォルト生成 | 事前登録必須 |

---

## 12. トレーシング（コード紹介）

> **Note**: Langfuseサーバーのセットアップが必要です。セルフホストで完全無料で運用できます。

```python
from graflow.trace.langfuse import LangFuseTracer
from graflow.core.workflow import workflow

# LangFuseTracer を作成（.env から自動で認証情報を読み込み）
tracer = LangFuseTracer(enable_runtime_graph=True)

with workflow("my_workflow", tracer=tracer) as wf:
    search >> analyze >> report
    wf.execute("search")
```

| 観点 | LangSmith（LangGraph） | Langfuse（Graflow） |
|---|---|---|
| **ライセンス** | クローズドソース（SaaS） | OSS（MIT License） |
| **セルフホスティング** | 不可 | Docker / ECS / Kubernetes |
| **コスト** | 有償プラン必須（本番利用） | セルフホストで完全無料 |
| **LLM対応** | LangChain経由のみ | LiteLLM対応の全プロバイダー |
| **分散トレース** | 非対応 | トレースID自動伝播 |

---

## まとめ: LangGraph vs Graflow 比較表

| 観点 | LangGraph | Graflow |
|---|---|---|
| **グラフ定義** | `add_node` + `add_edge` + `compile` | `>>` / `\|` 演算子（1行で構造が読める） |
| **データ共有** | State（TypedDict + Reducer） | Channel（Key-Value） + 自動キーワード引数解決 |
| **条件分岐** | `add_conditional_edges`（事前定義） | `next_task()` / `next_iteration()`（実行時動的） |
| **HITL** | `interrupt` + `Command(resume=)` | `request_feedback()` + タイムアウト時自動checkpoint |
| **チェックポイント** | 自動のみ | ユーザー制御（任意タイミングで保存） |
| **並列エラー制御** | なし | 4種の組み込みポリシー + カスタム |
| **分散実行** | なし（スレッド並列のみ） | Redisベースワーカーで水平スケール |
| **タスクハンドラー** | プロセス内のみ | direct / docker / カスタム |
| **LLM統合** | LangChainエコシステム前提 | LiteLLM + 任意のSuperAgentフレームワーク |
| **トレーシング** | LangSmith（有償SaaS） | Langfuse（OSS） + OpenTelemetry |
| **実行モデル** | Define-and-Run | Define-by-Run |

### リンク

- **GitHub**: [GraflowAI/graflow](https://github.com/GraflowAI/graflow)
- **プロジェクトサイト**: [graflow.ai](https://graflow.ai/)
- **ライセンス**: Apache 2.0